In [1]:
import os
import re
import glob
import subprocess
from IPython.display import HTML, display

# Ścieżka do narzędzia FlameGraph
FLAMEGRAPH_DIR = os.path.abspath("FlameGraph")
FLAMEGRAPH_PL = os.path.join(FLAMEGRAPH_DIR, "flamegraph.pl")

def clean_sym(s):
    """Upraszcza sygnatury C++ / Go dla czytelności Flame Graph."""
    s = s.strip()
    s = re.sub(r"\[clone .*?\]", "", s)
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"<.*?>", "", s)
    s = re.sub(r"\.abi0$", "", s)
    s = re.sub(r"\.func\d+$", "", s)
    s = s.replace(";", ":").replace(" ", "_")
    return s if s else "unknown"

class TreeNode:
    def __init__(self, name, pct=None):
        self.name = name
        self.pct = pct
        self.children = []

def parse_perf_report_tree(report_path):
    """Parsuje pełne hierarchiczne drzewo wywołań z perf_report.txt."""
    with open(report_path, "r", encoding="utf-8") as f:
        content = f.read()

    entries = []
    current_entry = None
    
    for line in content.split("\n"):
        m = re.match(r"^\s*([\d\.]+)%\s+([\d\.]+)%\s+(\S+)\s+(\S+)\s+\[.\]\s+(.*)", line)
        if m:
            if current_entry:
                entries.append(current_entry)
            current_entry = {
                "children": float(m.group(1)),
                "self": float(m.group(2)),
                "sym": clean_sym(m.group(5)),
                "lines": []
            }
        elif current_entry is not None:
            if line.strip().startswith("#"):
                continue
            current_entry["lines"].append(line)
            
    if current_entry:
        entries.append(current_entry)

    # Bierzemy główny korzeń programu (rekord o najwyższym udziale procentowym dzieci)
    root_entries = [e for e in entries if e["children"] > 30.0]
    if not root_entries:
        root_entries = entries[:1]
        
    root_data = root_entries[0]
    tree_lines = [l for l in root_data["lines"] if l.strip() and l.strip() != "|"]
    
    root_node = TreeNode(root_data["sym"], root_data["children"])
    stack = [(-1, root_node)]
    
    for line in tree_lines:
        bm = re.search(r"^(\s*)[\|\s]*(\-\-+)([\d\.]+)?%?(\-\-+)?\s*(.*)", line)
        if bm:
            col = len(bm.group(1)) + len(bm.group(2))
            pct = float(bm.group(3)) if bm.group(3) else None
            node_sym = clean_sym(bm.group(5))
            if not node_sym or node_sym == "..." or node_sym.startswith("|"):
                continue
            if len(stack) == 1 and node_sym == root_node.name:
                continue
                
            new_node = TreeNode(node_sym, pct)
            while len(stack) > 1 and stack[-1][0] >= col:
                stack.pop()
                
            stack[-1][1].children.append(new_node)
            stack.append((col, new_node))
        else:
            cm = re.search(r"^(\s*)([A-Za-z0-9_\.\:\*]+.*)", line)
            if cm:
                col = len(cm.group(1))
                node_sym = clean_sym(cm.group(2))
                if node_sym and not node_sym.startswith("|") and node_sym != "...":
                    new_node = TreeNode(node_sym, None)
                    stack[-1][1].children.append(new_node)
                    stack.append((col, new_node))

    folded_lines = []
    
    def emit_stacks(node, current_path):
        path = current_path + [node.name]
        node_pct = node.pct if node.pct is not None else 0.0
        
        children_sum = 0.0
        for ch in node.children:
            if ch.pct is not None:
                children_sum += ch.pct
            else:
                rem = max(0.0, node_pct - children_sum)
                children_sum += rem
                ch.pct = rem
                
        self_pct = max(0.0, node_pct - children_sum)
        if self_pct > 0.05:
            chain = ";".join(path)
            folded_lines.append(f"{chain} {int(round(self_pct * 100))}")
            
        for ch in node.children:
            emit_stacks(ch, path)
            
    emit_stacks(root_node, [])
    return folded_lines

def generate_flamegraph_from_report(report_path, output_svg_path, title="Flame Graph"):
    """Generuje interaktywny Flame Graph SVG z pliku perf_report.txt."""
    if not os.path.exists(report_path):
        print(f"Brak pliku: {report_path}")
        return False
        
    os.makedirs(os.path.dirname(output_svg_path), exist_ok=True)
    folded = parse_perf_report_tree(report_path)
    if not folded:
        print(f"Brak danych w {report_path}")
        return False
        
    folded_str = "\n".join(folded)
    p = subprocess.Popen(["perl", FLAMEGRAPH_PL, "--title", title, "--width", "1200"],
                         stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
    svg_output, _ = p.communicate(input=folded_str)
    
    with open(output_svg_path, "w", encoding="utf-8") as f:
        f.write(svg_output)
        
    print(f"Wygenerowano: {output_svg_path} ({len(folded)} gałęzi stosu)")
    return True

def show_flamegraph(svg_path):
    """Wyświetla interaktywny Flame Graph osadzony w notatniku."""
    if not os.path.exists(svg_path):
        print(f"Plik {svg_path} nie istnieje!")
        return
        
    with open(svg_path, "r", encoding="utf-8") as f:
        svg_content = f.read()
        
    html_code = f'''
    <div style="border: 1px solid #ddd; border-radius: 6px; padding: 10px; background: #fff; margin: 15px 0; overflow-x: auto;">
        {svg_content}
    </div>
    '''
    display(HTML(html_code))


In [2]:
base_logs = "hpc_logs" if os.path.exists("hpc_logs") else "plots/hpc_logs"

records = {
    "C++ OpenMP (noinline - 1 Wątek)": (
        f"{base_logs}/C-OMP/RECORD/C_OMP_RECORD/edupic_data/perf_report.txt",
        "flamegraphs_output/c_omp_noinline.svg",
        "C++ OpenMP (noinline) - 1 Wątek (Null-Collision)"
    ),
    "C++ OpenMP (8 Wątków)": (
        f"{base_logs}/C-OMP/RECORD/C_OMP_RECORD-8/edupic_data/perf_report.txt",
        "flamegraphs_output/c_omp_8t.svg",
        "C++ OpenMP - 8 Wątków (Null-Collision)"
    ),
    "Go Chunking (1 Core, 1 Worker)": (
        f"{base_logs}/Go-Chunking/RECORD/CHUNKING_RECORD-1-1/edupic_data/perf_report.txt",
        "flamegraphs_output/go_chunking_1_1.svg",
        "Go Chunking - 1 Rdzeń, 1 Gorutyna"
    ),
    "Go Chunking (8 Cores, 8 Workers)": (
        f"{base_logs}/Go-Chunking/RECORD/CHUNKING_RECORD-8-8/edupic_data/perf_report.txt",
        "flamegraphs_output/go_chunking_8_8.svg",
        "Go Chunking - 8 Rdzeni, 8 Gorutyn"
    ),
    "Go Channels (1 Core, 1 Worker)": (
        f"{base_logs}/Go-Channels/RECORD/CHANNELS_RECORD-1-1/edupic_data/perf_report.txt",
        "flamegraphs_output/go_channels_1_1.svg",
        "Go Channels - 1 Rdzeń, 1 Worker"
    ),
    "Go Channels (1 Core, 8 Workers)": (
        f"{base_logs}/Go-Channels/RECORD/CHANNELS_RECORD-1-8/edupic_data/perf_report.txt",
        "flamegraphs_output/go_channels_1_8.svg",
        "Go Channels - 1 Rdzeń, 8 Workerów (Oversubscription)"
    ),
}

print("Generowanie wykresów Flame Graph...")
for name, (rep_file, svg_file, title) in records.items():
    generate_flamegraph_from_report(rep_file, svg_file, title=title)


Generowanie wykresów Flame Graph...
Wygenerowano: flamegraphs_output/c_omp_noinline.svg (11 gałęzi stosu)
Wygenerowano: flamegraphs_output/c_omp_8t.svg (3 gałęzi stosu)
Wygenerowano: flamegraphs_output/go_chunking_1_1.svg (20 gałęzi stosu)
Wygenerowano: flamegraphs_output/go_chunking_8_8.svg (15 gałęzi stosu)
Wygenerowano: flamegraphs_output/go_channels_1_1.svg (12 gałęzi stosu)
Wygenerowano: flamegraphs_output/go_channels_1_8.svg (14 gałęzi stosu)


In [3]:
print("=== C++ OpenMP (noinline - 1 Wątek) ===")
show_flamegraph("flamegraphs_output/c_omp_noinline.svg")

print("=== C++ OpenMP (8 Wątków) ===")
show_flamegraph("flamegraphs_output/c_omp_8t.svg")


=== C++ OpenMP (noinline - 1 Wątek) ===


=== C++ OpenMP (8 Wątków) ===


## 2. Go Chunking — Flame Graphs
- **1 Rdzeń, 1 Gorutyna:** podział czasu między `Step3MoveElectrons` (~33%), `Step1ComputeElectronDensity` (~19%), `Step5CheckBoundariesElectrons` (~15%) i `Step7CollisionsElectrons` (~10%).
- **8 Rdzeni, 8 Gorutyn:** równomierne skalowanie przesuwania cząstek i zderzeń.


In [4]:
print("=== Go Chunking (1 Rdzeń, 1 Gorutyna) ===")
show_flamegraph("flamegraphs_output/go_chunking_1_1.svg")

print("=== Go Chunking (8 Rdzeni, 8 Gorutyn) ===")
show_flamegraph("flamegraphs_output/go_chunking_8_8.svg")


=== Go Chunking (1 Rdzeń, 1 Gorutyna) ===


=== Go Chunking (8 Rdzeni, 8 Gorutyn) ===


## 3. Go Channels — Flame Graphs
- **1 Worker:** główna pętla `startWorker` wykonuje zderzenia i ruch cząstek.
- **8 Workerów (1 Rdzeń OS):** widoczny narzut kolejkowania i przełączania kontekstu gorutyn.


In [5]:
print("=== Go Channels (1 Rdzeń, 1 Worker) ===")
show_flamegraph("flamegraphs_output/go_channels_1_1.svg")

print("=== Go Channels (1 Rdzeń, 8 Workerów) ===")
show_flamegraph("flamegraphs_output/go_channels_1_8.svg")


=== Go Channels (1 Rdzeń, 1 Worker) ===


=== Go Channels (1 Rdzeń, 8 Workerów) ===
